# df.pipe: feature engineering jako seria przekształceń

`df.pipe(fn)` to odpowiednik Unix pipes dla DataFrames:

```
cat data.csv | grep "delivered" | awk ... | sort
df.pipe(filter_delivered).pipe(add_features).pipe(model_input_prep)
```

Każda funkcja:
- przyjmuje `df` jako pierwszy argument
- zwraca nowy `df`
- nie ma efektów ubocznych (`df.copy()` na wejściu)

Zaleta dla testowania: każdy krok jest czystą funkcją → testujesz go z małym DataFrame.

## Problem: zagnieżdżone wywołania są nieczytelne

In [ ]:
import pandas as pd
import numpy as np

# Zagnieżdżone wywołania: trudno czytać, trudno testować
# model_input_prep(feature_engineering(merge_datasets(orders, items), payments))
# Kolejność? Co zwraca co? Nie wiadomo bez śledzenia nawiasów.
print("Zagnieżdżone wywołania czytamy od środka na zewnątrz, odwrotnie niż dane płyną.")
print("pipe() czytamy od góry do dołu, zgodnie z przepływem danych.")

## Dane: miniaturowy zbiór zamówień (Olist-style)

In [ ]:
orders = pd.DataFrame({
    "order_id":       ["A", "B", "C", "D"],
    "customer_id":    [1, 2, 3, 4],
    "purchase_ts":    pd.to_datetime(["2021-01-10 08:00", "2021-02-15 22:30",
                                      "2021-03-20 14:00", "2021-04-05 11:00"]),
    "delivery_ts":    pd.to_datetime(["2021-01-17", "2021-02-25",
                                      "2021-04-01", "2021-04-10"]),
    "estimated_ts":   pd.to_datetime(["2021-01-18", "2021-02-20",
                                      "2021-03-28", "2021-04-08"]),
})

items = pd.DataFrame({
    "order_id":      ["A", "B", "C", "D"],
    "price":         [100.0, 50.0, 200.0, 30.0],
    "freight_value": [10.0,   8.0,  20.0,  5.0],
    "items_count":   [2, 1, 3, 1],
    "category":      ["electronics", "fashion", "electronics", "fashion"],
})

payments = pd.DataFrame({
    "order_id":             ["A", "B", "C", "D"],
    "payment_type":         ["credit_card", "boleto", "credit_card", "voucher"],
    "payment_installments": [3, 1, 6, 1],
    "payment_value":        [110.0, 58.0, 220.0, 35.0],
})

print("orders:", orders.shape, "| items:", items.shape, "| payments:", payments.shape)

## Krok 1: merge_datasets

Konwencja: pierwszy argument to `df`, reszta to dodatkowe źródła danych.

In [ ]:
def merge_datasets(df, items, payments):
    df = df.copy()
    df = df.merge(items,    on="order_id", how="left")
    df = df.merge(payments, on="order_id", how="left")
    return df

## Krok 2: feature_engineering

`feature_engineering` sam wewnętrznie używa `pipe`, każda reguła FE to osobna funkcja.

In [ ]:
def _add_time_features(df):
    df = df.copy()
    df["delivery_delay_days"] = (df["delivery_ts"] - df["estimated_ts"]).dt.days
    df["purchase_hour"]       = df["purchase_ts"].dt.hour
    df["is_late"]             = (df["delivery_delay_days"] > 0).astype(int)
    return df

def _add_price_features(df):
    df = df.copy()
    df["freight_ratio"]    = df["freight_value"] / df["price"].clip(lower=1.0)
    df["price_per_item"]   = df["price"] / df["items_count"].clip(lower=1)
    return df

def _add_payment_features(df):
    df = df.copy()
    df["avg_installment_value"] = (
        df["payment_value"] / df["payment_installments"].clip(lower=1)
    )
    df["is_installment"] = (df["payment_installments"] > 1).astype(int)
    return df

def feature_engineering(df):
    return (
        df
        .pipe(_add_time_features)
        .pipe(_add_price_features)
        .pipe(_add_payment_features)
    )

## Krok 3: model_input_prep

Wybór kolumn + opcjonalne skalowanie.

In [ ]:
FEATURE_COLS = [
    "delivery_delay_days", "purchase_hour", "is_late",
    "freight_ratio", "price_per_item",
    "avg_installment_value", "is_installment",
    "items_count", "price",
]

def model_input_prep(df, feature_cols=FEATURE_COLS, scaler=None):
    X = df[feature_cols].copy()
    if scaler is not None:
        X[feature_cols] = scaler.transform(X)
    return X

## Pełny pipeline: trzy kroki, czytelne jak zdanie

In [ ]:
result = (
    orders
    .pipe(merge_datasets, items=items, payments=payments)
    .pipe(feature_engineering)
    .pipe(model_input_prep)
)

print(result.to_string())

## Z skalowaniem: scaler fitowany osobno na train

Model input prep działa tak samo dla train i test, zero duplikacji kodu.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Symulacja: "train" to pierwsze 3 zamówienia, "test" to 4.
orders_train = orders.iloc[:3]
orders_test  = orders.iloc[3:]

def prepare(df):
    return (
        df
        .pipe(merge_datasets, items=items, payments=payments)
        .pipe(feature_engineering)
    )

train_full = prepare(orders_train)
test_full  = prepare(orders_test)

scaler = StandardScaler().fit(train_full[FEATURE_COLS])

X_train = train_full.pipe(model_input_prep, scaler=scaler)
X_test  = test_full.pipe(model_input_prep,  scaler=scaler)

print("X_train:")
print(X_train.round(2).to_string())
print()
print("X_test (skalowany wg statystyk train):")
print(X_test.round(2).to_string())

## Testowanie: każda funkcja pipe jest czystą funkcją

Ponieważ każdy krok przyjmuje `df` i zwraca `df`, możemy testować go w izolacji
z minimalnym DataFrame. Nie potrzeba całego pipeline.

In [ ]:
# Test _add_time_features
def test_delivery_delay():
    df = pd.DataFrame({
        "delivery_ts":  pd.to_datetime(["2021-01-17", "2021-02-15"]),
        "estimated_ts": pd.to_datetime(["2021-01-15", "2021-02-20"]),
        "purchase_ts":  pd.to_datetime(["2021-01-10 09:00", "2021-02-01 23:00"]),
    })
    result = _add_time_features(df)
    assert result["delivery_delay_days"].tolist() == [2, -5], "delay błędny"
    assert result["is_late"].tolist()             == [1,  0], "is_late błędny"
    assert result["purchase_hour"].tolist()       == [9, 23], "purchase_hour błędny"
    print("✅ test_delivery_delay passed")

test_delivery_delay()

In [ ]:
# Test _add_price_features: wykrywa edge case: price = 0
def test_freight_ratio_zero_price():
    df = pd.DataFrame({
        "price":         [100.0, 0.0],    # 0.0 = edge case
        "freight_value": [10.0, 5.0],
        "items_count":   [2, 1],
    })
    result = _add_price_features(df)
    # clip(lower=1) chroni przed dzieleniem przez 0
    assert result["freight_ratio"].iloc[1] == 5.0, "powinno byc 5/1=5, nie inf"
    assert result["price_per_item"].iloc[1] == 0.0
    print("✅ test_freight_ratio_zero_price passed")

test_freight_ratio_zero_price()

In [ ]:
# Test niezmienności: pipe nie modyfikuje oryginalnego df
def test_no_mutation():
    df = pd.DataFrame({
        "delivery_ts":  pd.to_datetime(["2021-01-17"]),
        "estimated_ts": pd.to_datetime(["2021-01-15"]),
        "purchase_ts":  pd.to_datetime(["2021-01-10 10:00"]),
    })
    cols_before = set(df.columns)
    _add_time_features(df)          # wywołanie
    assert set(df.columns) == cols_before, "funkcja zmodyfikowała oryginalny df!"
    print("✅ test_no_mutation passed")

test_no_mutation()

## Porównanie: zagnieżdżone wywołania vs pipe

```python
# ❌ Bez pipe: czytasz od środka na zewnątrz
X = model_input_prep(
        feature_engineering(
            merge_datasets(orders, items=items, payments=payments)
        ),
        scaler=scaler
    )

# ✅ Z pipe: czytasz od góry do dołu, zgodnie z przepływem
X = (
    orders
    .pipe(merge_datasets,  items=items, payments=payments)
    .pipe(feature_engineering)
    .pipe(model_input_prep, scaler=scaler)
)
```

**Kolejna zaleta**: wystarczy zakomentować jeden krok żeby go wyłączyć.
Bez pipe musiałbyś rozwijać zagnieżdżone wywołania.